# 4. Threshold and the recall trade-off

**Question.** Missing a person with heart disease is worse than a false alarm. If the rule must catch
at least 90% of cases, what threshold does that need, what share of people does it flag, and how many
false alarms does each case found cost?

The threshold is picked on the **validation** cycle only and then applied unchanged to the test cycle.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from nhanes import nb, plots
from nhanes.features import CATEGORICAL, FEATURES, LABELS, NUMERIC, TARGET

ctx = nb.context()
df, R = ctx.df, ctx.R
pct = nb.pct
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 160)
print("mode:", "synthetic fixture (CI)" if ctx.fixture else "real NHANES data")
print(f"{len(df):,} adults, {int(df[TARGET].sum()):,} reporting heart disease")

mode: real NHANES data
22,512 adults, 1,637 reporting heart disease


## 4.1 Picking the threshold on validation data (code, not a lookup)

In [2]:
from nhanes.metrics import threshold_for_recall, threshold_metrics

sel = R["selection"]["selected"]
pred = ctx.pred
valid, test = pred[pred.split == "valid"], pred[pred.split == "test"]
yv, pv = valid[TARGET].to_numpy(), valid[f"p_{sel}"].to_numpy()
yt, pt = test[TARGET].to_numpy(), test[f"p_{sel}"].to_numpy()
t90 = threshold_for_recall(yv, pv, 0.90)
nb.check("90% recall threshold", t90, R["models"][sel]["recall_target"]["threshold"])
mv, mt = threshold_metrics(yv, pv, t90), threshold_metrics(yt, pt, t90)
print(
    f"Selected model: {sel}. Threshold on validation for 90% recall: {pct(t90, 2)} predicted risk"
)
print(
    f"Validation recall {pct(mv['recall'])}, then on the test cycle: recall {pct(mt['recall'])}, precision {pct(mt['precision'])}, share flagged {pct(mt['flagged_share'])}"
)

Check: 90% recall threshold: notebook 0.0391077 vs results.json 0.0391077 (matches)
Selected model: logistic_regression. Threshold on validation for 90% recall: 3.91% predicted risk
Validation recall 90.2%, then on the test cycle: recall 91.2%, precision 16.0%, share flagged 47.2%


## 4.2 The whole trade-off curve

In [3]:
nb.show(plots.plot_threshold_tradeoff(yv, pv, yt, pt))
trade = pd.DataFrame(
    [
        {
            "target recall": t["target_recall"],
            "threshold": t["threshold"],
            "valid recall": t["valid"]["recall"],
            "test recall": t["test"]["recall"],
            "test precision": t["test"]["precision"],
            "share flagged": t["test"]["flagged_share"],
            "false alarms per case found": t["test"]["false_alarms_per_case_found"],
        }
        for t in R["threshold_tradeoff_selected"]
    ]
).set_index("target recall")
display(trade)

<Figure size 1320x517 with 2 Axes>

,threshold,valid recall,test recall,test precision,share flagged,false alarms per case found
target recall,,,,,,
0.800,0.075,0.801,0.818,0.201,0.337,3.984
0.850,0.060,0.850,0.869,0.188,0.382,4.320
0.900,0.039,0.902,0.912,0.160,0.472,5.269
0.950,0.024,0.951,0.954,0.135,0.582,6.390


## 4.3 What 1,000 screened adults look like

In [4]:
scale = 1000 / len(yt)
per1000 = (
    pd.Series(
        {
            "adults screened": 1000,
            "with heart disease (self-reported)": mt["tp"] * scale + mt["fn"] * scale,
            "cases caught": mt["tp"] * scale,
            "cases missed": mt["fn"] * scale,
            "flagged in total": (mt["tp"] + mt["fp"]) * scale,
            "false alarms": mt["fp"] * scale,
        }
    )
    .round(0)
    .astype(int)
)
display(per1000.to_frame("per 1,000 adults"))

,"per 1,000 adults"
adults screened,1000
with heart disease (self-reported),83
cases caught,75
cases missed,7
flagged in total,472
false alarms,397


## 4.4 If the cost of a miss is a known multiple of a false alarm

Suppose one missed case is as bad as *r* false alarms. For each *r*, choose the threshold that
minimises expected cost **on validation**, then report what it does on the test cycle.


In [5]:
grid = np.unique(np.quantile(pv, np.linspace(0, 0.999, 400)))
rows = []
for r in (1, 5, 10, 20, 50):
    costs = [
        r * threshold_metrics(yv, pv, t)["fn"] + threshold_metrics(yv, pv, t)["fp"] for t in grid
    ]
    t = float(grid[int(np.argmin(costs))])
    m = threshold_metrics(yt, pt, t)
    rows.append(
        {
            "miss costs r false alarms": r,
            "threshold": t,
            "test recall": m["recall"],
            "share flagged": m["flagged_share"],
            "false alarms per case found": m["false_alarms_per_case_found"],
        }
    )
costtab = pd.DataFrame(rows).set_index("miss costs r false alarms")
display(costtab)

,threshold,test recall,share flagged,false alarms per case found
miss costs r false alarms,,,,
1,0.638,0.002,0.001,2.000
5,0.121,0.672,0.230,3.150
10,0.072,0.823,0.344,4.064
20,0.055,0.877,0.402,4.549
50,0.020,0.958,0.621,6.842


## 4.5 Who gets flagged: subgroups at the 90% recall threshold

In [6]:
nb.show(plots.plot_subgroups(R["subgroups_selected"]))
sub = pd.DataFrame(R["subgroups_selected"]["age_band"]).set_index("group")[
    ["n", "positives", "prevalence", "recall", "precision", "flagged_share"]
]
display(sub)
sub_sex = pd.DataFrame(R["subgroups_selected"]["sex"]).set_index("group")[
    ["n", "positives", "prevalence", "recall", "precision", "flagged_share"]
]
display(sub_sex)

<Figure size 1320x528 with 2 Axes>

,n,positives,prevalence,recall,precision,flagged_share
group,,,,,,
20-39,1684,17,0.010,0.176,0.176,0.010
40-59,1723,76,0.044,0.711,0.090,0.348
60+,2127,364,0.171,0.989,0.180,0.939


,n,positives,prevalence,recall,precision,flagged_share
group,,,,,,
female,2854,159,0.056,0.862,0.115,0.416
male,2680,298,0.111,0.940,0.196,0.532


## What this means

In [7]:
lo = sub.recall.idxmin()
hi = sub.recall.idxmax()
nb.say(
    f"- To catch {pct(0.90, 0)} of cases on validation the rule must flag everyone with predicted risk of at least {pct(t90, 1)}. "
    f"On the test cycle that caught {pct(mt['recall'])} of cases while flagging {pct(mt['flagged_share'])} of all adults, "
    f"which is {mt['fp'] / mt['tp']:.1f} false alarms for every case found.\n"
    f"- Per 1,000 adults: about {int(per1000['flagged in total'])} flagged, {int(per1000['cases caught'])} cases caught and "
    f"{int(per1000['cases missed'])} missed.\n"
    f"- Raising the target from 80% to 95% moves false alarms per case found from "
    f"{trade['false alarms per case found'].iloc[0]:.1f} to {trade['false alarms per case found'].iloc[-1]:.1f}.\n"
    f"- The rule treats age bands very differently: recall is {pct(sub.recall[hi])} in the {hi} band (with {pct(sub.flagged_share[hi])} of that band flagged) "
    f"and {pct(sub.recall[lo])} in the {lo} band ({int(sub.positives[lo])} cases)."
)

- To catch 90% of cases on validation the rule must flag everyone with predicted risk of at least 3.9%. On the test cycle that caught 91.2% of cases while flagging 47.2% of all adults, which is 5.3 false alarms for every case found.
- Per 1,000 adults: about 472 flagged, 75 cases caught and 7 missed.
- Raising the target from 80% to 95% moves false alarms per case found from 4.0 to 6.4.
- The rule treats age bands very differently: recall is 98.9% in the 60+ band (with 93.9% of that band flagged) and 17.6% in the 20-39 band (17 cases).

## What is weak

In [8]:
nb.say(
    f"- Validation recall was {pct(mv['recall'])} and test recall {pct(mt['recall'])} (a difference of {(mt['recall'] - mv['recall']) * 100:+.1f} points). "
    f"The threshold is not guaranteed to carry over to a later cycle: prevalence was {pct(yv.mean())} on validation and {pct(yt.mean())} on test, "
    "so in a new period recall can land below the target.\n"
    f"- The overall target is met partly by how the rule treats the oldest band: {pct(sub.flagged_share[hi])} of the {hi} band is flagged, "
    f"against {pct(sub.flagged_share[lo])} of the {lo} band. One global threshold on a model dominated by age behaves like an age cut-off, "
    "and that has a high false alarm cost.\n"
    "- The cost ratios in 4.4 are illustrations. The data contain no costs; a real deployment would need real ones.\n"
    "- Small subgroups (see the case counts) give recall figures that move a lot with a handful of people."
)

- Validation recall was 90.2% and test recall 91.2% (a difference of +1.1 points). The threshold is not guaranteed to carry over to a later cycle: prevalence was 7.5% on validation and 8.3% on test, so in a new period recall can land below the target.
- The overall target is met partly by how the rule treats the oldest band: 93.9% of the 60+ band is flagged, against 1.0% of the 20-39 band. One global threshold on a model dominated by age behaves like an age cut-off, and that has a high false alarm cost.
- The cost ratios in 4.4 are illustrations. The data contain no costs; a real deployment would need real ones.
- Small subgroups (see the case counts) give recall figures that move a lot with a handful of people.